# 05 — Transformer para Classificação de EEG

Este notebook treina um **Transformer Encoder-Only** para classificar estados mentais a partir das features espectrais do EEG.

## Estratégia: Canal como token

Diferente da CNN (que trata as 25 features como uma grade espacial 5×5) e da LSTM (que trata as bandas como sequência temporal), o Transformer usa **cada canal EEG como um token**:

```
(N, 25) → reshape → (N, 5_canais, 5_bandas)
                         ↑ tokens     ↑ features por token
         AF3, AF4, T7, T8, Pz
```

O mecanismo de **self-attention** aprende correlações entre os 5 eletrodos:
- **AF3 ↔ AF4**: frontais simétricos
- **T7 ↔ T8**: temporais simétricos — principais em motor imagery
- **Pz**: parietal — comportamento distinto nos outros canais

## Arquitetura

```
Input(5, 5)
  → Dense(d_model=64)            # embedding linear por token
  → Learnable Positional Emb.    # identidade de cada canal
  → Prepend [CLS] token          # sequência: 6 tokens
  → 2× TransformerEncoderBlock   # pre-norm: LN → MHA → Add, LN → FFN → Add
  → Extrai [CLS]                 # representação global
  → LN → Dropout → Dense(3)      # classificação
```

Os dados utilizados são os gerados pelo notebook `02_preprocessamento_base.ipynb`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

from src.models.transformer import TransformerConfig, build_transformer_model, train_transformer

## Carregar dados pré-processados

In [ ]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

X_train = np.load(PROCESSED_DIR / "X_train_base.npy")
X_test  = np.load(PROCESSED_DIR / "X_test_base.npy")
y_train = np.load(PROCESSED_DIR / "y_train.npy")
y_test  = np.load(PROCESSED_DIR / "y_test.npy")

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)

---
## Treinamento do Transformer

### Hiperparâmetros

| Parâmetro | Valor | Justificativa |
|-----------|-------|---------------|
| `d_model` | 64 | Proporcional ao tamanho do token (5 features) — não inflar |
| `num_heads` | 4 | Power-of-2, divide d_model — 4 perspectivas de atenção entre canais |
| `num_layers` | 2 | Sequência curta (5 tokens) não necessita de profundidade |
| `ff_dim` | 128 | 2×d_model — padrão conservador |
| `dropout_rate` | 0.1 | Mais leve que CNN/LSTM — attention já regulariza |
| `learning_rate` | 0.0003 | Transformers são sensíveis a lr alta |
| `weight_decay` | 1e-4 | AdamW — regularização L2 desacoplada, padrão Transformers |
| `patience` | 15 | Maior que CNN/LSTM — Transformers convergem mais lento |

In [ ]:
transformer_config = TransformerConfig()

transformer_result = train_transformer(X_train, y_train, X_test, y_test, transformer_config)

### Resumo da arquitetura

In [ ]:
transformer_result.model.summary()

### Curvas de aprendizado — Transformer

- Linha **azul** (train): desempenho nos dados de treino.
- Linha **laranja** (val): desempenho nos dados de validação (20% do treino).

Separação grande entre as linhas indica overfitting.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs_ran = range(1, len(transformer_result.history["loss"]) + 1)

ax1.plot(epochs_ran, transformer_result.history["loss"], label="Treino")
ax1.plot(epochs_ran, transformer_result.history["val_loss"], label="Validação")
ax1.set_title("Loss — Transformer")
ax1.set_xlabel("Época")
ax1.set_ylabel("Loss")
ax1.legend()

ax2.plot(epochs_ran, transformer_result.history["accuracy"], label="Treino")
ax2.plot(epochs_ran, transformer_result.history["val_accuracy"], label="Validação")
ax2.set_title("Acurácia — Transformer")
ax2.set_xlabel("Época")
ax2.set_ylabel("Acurácia")
ax2.legend()

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "figures" / "transformer_learning_curves.png", dpi=150)
plt.show()

### Resultado no conjunto de teste — Transformer

In [ ]:
print(f"Loss no teste:     {transformer_result.test_loss:.4f}")
print(f"Acurácia no teste: {transformer_result.test_accuracy:.4f} ({transformer_result.test_accuracy * 100:.2f}%)")

### Matriz de confusão — Transformer

In [ ]:
cm = confusion_matrix(transformer_result.y_test, transformer_result.y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Esquerda", "Direita", "Neutro"])
disp.plot(ax=ax, colorbar=False)
ax.set_title("Matriz de Confusão — Transformer")

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "figures" / "transformer_confusion_matrix.png", dpi=150)
plt.show()

### Relatório de classificação — Transformer

- **Precision**: de tudo que o modelo disse ser classe X, quantos eram de fato classe X.
- **Recall**: de todas as amostras da classe X, quantas o modelo acertou.
- **F1-score**: média harmônica entre precision e recall.

In [ ]:
print(classification_report(
    transformer_result.y_test,
    transformer_result.y_pred,
    target_names=["Esquerda", "Direita", "Neutro"],
))

---
## Visualização das Attention Weights

O Transformer aprende **pesos de atenção** entre os 5 canais EEG. Visualizar a matriz de atenção média revela quais pares de eletrodos o modelo considera mais correlacionados para a classificação.

> **Nota**: A atenção é extraída da **primeira camada** (padrões mais básicos). Cada head representa uma "perspectiva" diferente sobre as relações entre canais.

In [ ]:
import tensorflow as tf
from src.models.transformer import reshape_for_transformer

CHANNEL_NAMES = ["AF3", "AF4", "T7", "T8", "Pz"]

X_test_t = reshape_for_transformer(X_test)

# Sub-model: input → output of LayerNorm just before first MHA
ln_layers  = [l for l in transformer_result.model.layers if "layer_normalization" in l.name]
mha_layers = [l for l in transformer_result.model.layers if "multi_head_attention"  in l.name]

pre_attn_model = tf.keras.Model(
    inputs=transformer_result.model.input,
    outputs=ln_layers[0].output,  # (batch, 6, d_model) — input to first MHA
)

pre_attn_out = pre_attn_model(X_test_t[:500], training=False)

# Call the first MHA layer with return_attention_scores=True to get (output, scores)
_, attn_scores = mha_layers[0](
    pre_attn_out, pre_attn_out,
    return_attention_scores=True,
    training=False,
)
attn_scores = attn_scores.numpy()  # (500, num_heads, 6, 6)

# Average over samples; extract 5×5 channel sub-matrix (skip CLS token at index 0)
attn_mean    = attn_scores.mean(axis=0)   # (num_heads, 6, 6)
attn_channel = attn_mean[:, 1:, 1:]       # (num_heads, 5, 5)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for h, ax in enumerate(axes):
    im = ax.imshow(attn_channel[h], cmap="Blues", vmin=0, vmax=attn_channel[h].max())
    ax.set_xticks(range(5))
    ax.set_yticks(range(5))
    ax.set_xticklabels(CHANNEL_NAMES, fontsize=8)
    ax.set_yticklabels(CHANNEL_NAMES, fontsize=8)
    ax.set_title(f"Head {h + 1}")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle("Attention weights médios — Camada 1 (canal × canal)", y=1.02)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "figures" / "transformer_attention_weights.png", dpi=150, bbox_inches="tight")
plt.show()

## Salvar modelo

In [ ]:
MODELS_DIR = PROJECT_ROOT / "outputs" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

transformer_result.model.save(MODELS_DIR / "transformer_model.keras")
print("Modelo salvo em outputs/models/transformer_model.keras")

## Exportar métricas para o dashboard

In [ ]:
from src.evaluation.export_metrics import export_metrics

users_test = np.load(PROCESSED_DIR / "users_test.npy", allow_pickle=True)
export_metrics(transformer_result, model_name="transformer", users_test=users_test)